In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/archive/ASA24_GPTFoodCodes_metrics_final_old.csv')

In [3]:
MATCH_KEY = ['FileName', 'PortionType', 'Energy (kcal)']
df_deduped = df.drop_duplicates(subset=MATCH_KEY).reset_index(drop=True)
df_deduped.shape

(413, 44)

In [4]:
df_deduped.to_csv('../data/archive/ASA24_GPTFoodCodes_metrics_final_old_deduped.csv', index=False)

In [8]:
df_main = pd.read_csv('/Volumes/My Passport/Asa24PortionImages/ASA24_ImageMetadata.csv')
df_lookup = pd.read_csv('/Volumes/My Passport/Asa24PortionImages/ASA24_ImageMetadata1.csv')

In [9]:
LOOKUP_KEY_COL = 'FileName'        
TARGET_COL = 'Portion'
# Uncapitalize all the values in LOOKUP_KEY_COL to match MAIN_KEY_COL
df_lookup[LOOKUP_KEY_COL] = df_lookup[LOOKUP_KEY_COL].str.lower()
df_lookup_subset = df_lookup[[LOOKUP_KEY_COL, TARGET_COL]]
# Merge the two dataframes on LOOKUP_KEY_COL
df_merged = pd.merge(df_main, df_lookup_subset, on=LOOKUP_KEY_COL, how='left')
output_file_path = '/Volumes/My Passport/Asa24PortionImages/ASA24_ImageMetadata.csv'
df_merged.to_csv(output_file_path, index=False)

In [3]:
df = pd.read_csv("/Users/ethanyao/Library/Mobile Documents/com~apple~CloudDocs/GPT Food Project/DietAI24/ASA24_GPTFoodCodes_all_portions.csv")

# Remove all rows where 'Link' is NA
df_cleaned = df[df['Link'].notna()].copy()
df_cleaned.reset_index(drop=True, inplace=True)
print(df_cleaned.shape)
df_cleaned.to_csv("/Users/ethanyao/Library/Mobile Documents/com~apple~CloudDocs/GPT Food Project/DietAI24/ASA24_GPTFoodCodes_all_portions.csv", index=False)

(1502, 10)


In [2]:
# Merge PortionCode and PortionSubCode from ASA24_ImageMetadata into GPTFoodCodes_portion

# Load results file (output from rag_portion_size.py)
df_results = pd.read_csv('../data/ASA24_GPTFoodCodes_portion1.csv')

# Load original metadata with PortionCode and PortionSubCode
df_metadata = pd.read_csv('/Volumes/My Passport/Asa24PortionImages/ASA24_ImageMetadata.csv')

# Check if columns already exist
if 'PortionCode' in df_results.columns and 'PortionSubCode' in df_results.columns:
    print("PortionCode and PortionSubCode already exist in the file")
else:
    # Merge on FileName, FC_Description, and Portion
    df_results = df_results.merge(
        df_metadata[['FileName', 'FC_Description', 'Portion', 'PortionCode', 'PortionSubCode']], 
        on=['FileName', 'FC_Description', 'Portion'], 
        how='left'
    )
    
    # Save updated file
    # df_results.to_csv('../ASA24_GPTFoodCodes_portion.csv', index=False)
    print(f"Added PortionCode and PortionSubCode columns")
    print(f"Result shape: {df_results.shape}")
    print(f"Columns: {df_results.columns.tolist()}")

PortionCode and PortionSubCode already exist in the file


In [9]:
df_results.drop(columns=['PortionCode', 'PortionSubCode'], inplace=True)

In [5]:
# Remove all duplicate rows based on all columns
df_results.drop_duplicates(inplace=True)
df_results.reset_index(drop=True, inplace=True)
print(f"After removing duplicates, shape: {df_results.shape}")

After removing duplicates, shape: (1715, 19)


In [6]:
df_results.to_csv('../ASA24_GPTFoodCodes_portion.csv', index=False)

In [38]:
df_results = pd.read_csv('../FoodWeights.csv')
df_metadata = pd.read_csv('/Volumes/My Passport/Asa24PortionImages/ASA24_ImageMetadata.csv')
# Remove NA values in FoodCode column of df_metadata
df_metadata = df_metadata[df_metadata['FoodCode'].notna()].copy()
# Make both types as int64
df_results['FoodCode'] = df_results['FoodCode'].astype('int64')
df_metadata['FoodCode'] = df_metadata['FoodCode'].astype('int64')
df_results = df_results.merge(
        df_metadata[['FoodCode', 'Portion', 'PortionSubCode', 'PortionCode']], 
        on=['FoodCode', 'Portion'], 
        how='left'
    )

In [39]:
df_results.shape

(26000, 9)

In [40]:
# Rename the column 'PortionSubCode' to 'SubCode'
df_results.rename(columns={'PortionSubCode': 'SubCode'}, inplace=True)

In [41]:
df_results.head()

,FoodCode,FC_Description,WWEIA Category number,WWEIA Category description,Seq num,Portion,Portion weight (g),SubCode,PortionCode
0,11000000,"Milk, human",9602,Human milk,1,1 cup,246.0,NaN,NaN
1,11000000,"Milk, human",9602,Human milk,2,Quantity not specified,0.0,NaN,NaN
2,11000000,"Milk, human",9602,Human milk,3,1 FO,30.8,NaN,NaN
3,11100000,"Milk, NFS",1004,"Milk, reduced fat",1,1 cup,244.0,0.0,10205.0
4,11100000,"Milk, NFS",1004,"Milk, reduced fat",2,1 FO,30.5,NaN,NaN


In [43]:
# find the missing values in the PortionSubCode and PortionCode columns
missing_subcode = df_results['SubCode'].isna().sum()
missing_code = df_results['PortionCode'].isna().sum()
print(f"Missing PortionSubCode: {missing_subcode}, Missing PortionCode: {missing_code}")

Missing PortionSubCode: 20977, Missing PortionCode: 20977


In [19]:
df_results.to_csv('../Food Weights.csv', index=False)

In [15]:
# =============================================================================
# Calculate Portion Weights for ASA24_GPTFoodCodes_portion.csv
# Uses FoodWeights.csv to look up standard portion weights and multiply by amount
# =============================================================================

import re

# Load the files
df_portion_results = pd.read_csv('../data/archive/ASA24_GPTFoodCodes_portion.csv')
df_weights = pd.read_csv('../data/FoodWeights.csv')

# Extract base unit from FoodWeights portions (e.g., "1 cup" -> "cup", "1 tablespoon" -> "tablespoon")
def extract_base_unit(portion):
    """Extract the base unit from a portion string like '1 cup' or '1 cup, cooked'"""
    if pd.isna(portion):
        return None
    portion = str(portion).lower().strip()
    match = re.match(r'^1\s+([a-z]+)', portion)
    if match:
        return match.group(1)
    return None

# Build the weight lookup table
# For each FoodCode + unit combination, keep the simplest (shortest) portion description
df_weights['BaseUnit'] = df_weights['Portion'].apply(extract_base_unit)
df_weights['Simplicity'] = df_weights['Portion'].apply(lambda x: len(str(x)) if pd.notna(x) else 999)

df_weights_simple = df_weights.dropna(subset=['BaseUnit']).sort_values('Simplicity')
df_weights_simple = df_weights_simple.drop_duplicates(subset=['FoodCode', 'BaseUnit'], keep='first')

weight_lookup = df_weights_simple[['FoodCode', 'BaseUnit', 'Portion weight (g)']].copy()

print(f"Weight lookup table: {len(weight_lookup)} entries")
print(f"Unique FoodCodes in lookup: {weight_lookup['FoodCode'].nunique()}")
print(f"Unique BaseUnits in lookup: {weight_lookup['BaseUnit'].nunique()}")

Weight lookup table: 14937 entries
Unique FoodCodes in lookup: 5572
Unique BaseUnits in lookup: 391


In [ ]:
# Normalize ASA24 LabelUnit to match FoodWeights BaseUnit
df_portion_results['BaseUnit'] = df_portion_results['LabelUnit'].str.lower().str.strip()

# Merge to get standard portion weights (ground truth unit)
df_portion_merged = df_portion_results.merge(
    weight_lookup,
    left_on=['FoodCode', 'BaseUnit'],
    right_on=['FoodCode', 'BaseUnit'],
    how='left'
)

# Calculate ground truth portion weight (uses LabelUnit + LabelAmount)
df_portion_merged['CalculatedWeight'] = (
    df_portion_merged['Portion weight (g)'] * df_portion_merged['LabelAmount']
)

# ---------------------------------------------------------------------------
# GPT weight calculation: use GPTPortionDescription unit + GPTPortionAmount
# ---------------------------------------------------------------------------

def extract_gpt_unit(description):
    """Extract normalized base unit from GPTPortionDescription.

    Handles any numeric prefix (not just '1') and strips plurals.
    Examples: '2 cups' -> 'cup', '1 tablespoon' -> 'tablespoon'
    Returns None for unrecognizable / failure strings.
    """
    if pd.isna(description):
        return None
    text = str(description).lower().strip()
    match = re.match(r'^\d+(?:\.\d+)?\s+([a-z]+)', text)
    if not match:
        return None
    unit = match.group(1)
    # Strip trailing 's' to normalise plurals (cups->cup, pieces->piece)
    # Avoid over-stripping short words like 'oz'
    if len(unit) > 3 and unit.endswith('s'):
        unit = unit[:-1]
    return unit

# Extract GPT's chosen unit from GPTPortionDescription
df_portion_merged['GPTBaseUnit'] = (
    df_portion_merged['GPTPortionDescription'].apply(extract_gpt_unit)
)

# Separate weight lookup using GPT's unit (not LabelUnit)
gpt_weight_lookup = weight_lookup.rename(columns={'Portion weight (g)': 'Portion weight (g)_GPT'})
df_portion_merged = df_portion_merged.merge(
    gpt_weight_lookup,
    left_on=['FoodCode', 'GPTBaseUnit'],
    right_on=['FoodCode', 'BaseUnit'],
    how='left',
    suffixes=('', '_gpt_lookup')
).drop(columns=['BaseUnit_gpt_lookup'], errors='ignore')

# GPT weight = GPT weight-per-unit x GPT predicted amount
df_portion_merged['CalculatedWeightGPT'] = (
    df_portion_merged['Portion weight (g)_GPT'] *
    pd.to_numeric(df_portion_merged['GPTPortionAmount'], errors='coerce')
)

# ---------------------------------------------------------------------------
# Match statistics
# ---------------------------------------------------------------------------
total_rows = len(df_portion_merged)
matched_rows = df_portion_merged['Portion weight (g)'].notna().sum()
unmatched_rows = df_portion_merged['Portion weight (g)'].isna().sum()
gpt_matched = df_portion_merged['Portion weight (g)_GPT'].notna().sum()

print(f"=== MATCH RESULTS ===")
print(f"Total rows:         {total_rows}")
print(f"GT unit matched:    {matched_rows} ({matched_rows/total_rows*100:.1f}%)")
print(f"GT unit unmatched:  {unmatched_rows} ({unmatched_rows/total_rows*100:.1f}%)")
print(f"GPT unit matched:   {gpt_matched} ({gpt_matched/total_rows*100:.1f}%)")
print(f"GPT unit unmatched: {total_rows - gpt_matched} ({(total_rows-gpt_matched)/total_rows*100:.1f}%)")

In [19]:
# Analyze unmatched cases - categorize the reasons
unmatched = df_portion_merged[df_portion_merged['Portion weight (g)'].isna()].copy()

if len(unmatched) > 0:
    # Check if FoodCode exists in FoodWeights
    all_food_codes = set(df_weights['FoodCode'].unique())
    unmatched['FoodCodeExists'] = unmatched['FoodCode'].isin(all_food_codes)
    
    # Categorize unmatched reasons
    fc_missing = (~unmatched['FoodCodeExists']).sum()
    unit_missing = (unmatched['FoodCodeExists']).sum()  # FoodCode exists but unit doesn't
    
    print(f"=== UNMATCHED ANALYSIS ===")
    print(f"FoodCode not in FoodWeights: {fc_missing} ({fc_missing/len(unmatched)*100:.1f}%)")
    print(f"FoodCode exists but unit not found: {unit_missing} ({unit_missing/len(unmatched)*100:.1f}%)")
    
    print(f"\n=== Top unmatched units (FoodCode exists but unit missing) ===")
    unit_missing_df = unmatched[unmatched['FoodCodeExists']]
    print(unit_missing_df['LabelUnit'].value_counts().head(15))
else:
    print("All rows matched successfully!")

=== UNMATCHED ANALYSIS ===
FoodCode not in FoodWeights: 323 (34.7%)
FoodCode exists but unit not found: 609 (65.3%)

=== Top unmatched units (FoodCode exists but unit missing) ===
LabelUnit
teaspoon                54
piece                   39
tablespoon              38
ounce                   29
roll                    25
cookie                  24
whole vegetable         17
cup                     16
pastry                  16
slice                   11
fruit                    8
egg roll                 8
individual container     8
medium size              7
link                     7
Name: count, dtype: int64


In [20]:
df_matched_only = df_portion_merged[df_portion_merged['Portion weight (g)'].notna()]
df_matched_only.head()

,Unnamed: 0,FileName,PortionType,FoodCode,Main Food description,Portion,Multiplier,Rank,PortionCode,PortionSubCode,...,LabelUnit,PortionShot,GPTPortionDescription,GPTPortionReason,GPTPortionAmount,GPTPortionAmountReason,BaseUnit,Portion weight (g),CalculatedWeight,CalculatedWeightGPT
0,0,cp_42fo.png,largest,94000100,"Water, tap",42 FO,42.0,0,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",16 FO,NaN,0,NaN,fo,30.0,1260.0,0.0
1,1,tcup_6fo.png,median,94000100,"Water, tap",6 FO,6.0,0,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",6 FO,NaN,0,NaN,fo,30.0,180.0,0.0
2,2,mug_2fo.png,smallest,94000100,"Water, tap",2 FO,0.1,0,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",I can't help to analyze this image.,"The mug's contents are not visible, so the por...",1,NaN,fo,30.0,60.0,30.0
3,3,cp_42fo.png,largest,94100100,"Water, bottled, plain",42 FO,42.0,1,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",8 FO,NaN,1,NaN,fo,30.0,1260.0,30.0
4,4,glsstall_12fo.png,median,94100100,"Water, bottled, plain",12 FO,6.0,1,30000,0,...,FO,"1 tablespoon ,2 tablespoons ,1/4 cup ,1/2 cup ...",I can't help to analyze this image.,Reason: the image shows an empty clear tall dr...,0,NaN,fo,30.0,360.0,0.0


In [21]:
df_matched_only.shape

(1055, 27)

In [13]:
df_matched_only.to_csv('../data/ASA24_GPTFoodCodes_portion_with_weights.csv', index=False)

In [6]:
# Preview the results with calculated weights
print("=== Sample rows with calculated weights ===")
display_cols = ['FoodCodeCommon', 'FC_Description', 'Portion', 'LabelAmount', 'LabelUnit', 
                'Portion weight (g)', 'CalculatedWeight', 'GPTPortionAmount', 'CalculatedWeightGPT']
df_portion_merged[display_cols].head(15)

=== Sample rows with calculated weights ===


,FoodCodeCommon,FC_Description,Portion,LabelAmount,LabelUnit,Portion weight (g),CalculatedWeight,GPTPortionAmount,CalculatedWeightGPT
0,91101010,"Sugar, white, granulated or lump",6 teaspoons,6.0,teaspoon,4.2,25.2,6,25.2
1,75113000,"Lettuce, raw",4 cups,4.0,cup,35.0,140.0,1,35.0
2,74401010,Ketchup,1 packet,1.0,packet,10.0,10.0,I can't help to analyze this image.,NaN
3,74101000,"Tomatoes, raw",6 pieces,6.0,piece,NaN,NaN,5,NaN
4,63101000,"Apple, raw",1 medium size,1.0,medium size,NaN,NaN,1,NaN
5,63101000,"Apple, raw",1 medium size,1.0,medium size,NaN,NaN,1,NaN
6,63101000,"Apple, raw",1 medium size,1.0,medium size,NaN,NaN,1,NaN
7,63101000,"Apple, raw",1 medium size,1.0,medium size,NaN,NaN,1,NaN
8,71401030,"Potato, french fries, fast food",2 cups,2.0,cup,60.0,120.0,1,60.0
9,83107000,"Mayonnaise, regular",6 tablespoons,6.0,tablespoon,13.8,82.8,6,82.8


In [7]:
# Clean up and save results
# Remove temporary columns
df_final = df_portion_merged.drop(columns=['BaseUnit', 'FoodCode'], errors='ignore')

print(f"Final dataframe shape: {df_final.shape}")
print(f"Columns: {df_final.columns.tolist()}")

# Uncomment to save
# df_final.to_csv('../ASA24_GPTFoodCodes_portion_with_weights.csv', index=False)
# print("Saved to ../ASA24_GPTFoodCodes_portion_with_weights.csv")

Final dataframe shape: (1715, 22)
Columns: ['FileName', 'PortionType', 'FoodCodeCommon', 'FC_Description', 'Portion', 'Multiplier', 'Rank', 'Link', 'GPTFoodDescription', 'GPTFoodCode', 'LabelAmount', 'LabelUnit', 'PortionShot', 'GPTPortionDescription', 'GPTPortionReason', 'GPTPortionAmount', 'GPTPortionAmountReason', 'PortionCode', 'PortionSubCode', 'Portion weight (g)', 'CalculatedWeight', 'CalculatedWeightGPT']


In [8]:
# Replace this with your actual results file path
file_path = '../ASA24_GPTFoodCodes_portion.csv' 

print(f"Loading data from: {file_path}")
df = pd.read_csv(file_path)
print(f"Total rows in file: {len(df)}")
print("-" * 50)

# ==============================================================================
# 1. Check Predictions (Missing GPT Descriptions)
# ==============================================================================
# We check for NaN, None, and also empty strings just in case
missing_gpt = df['GPTPortionDescription'].isna() | (df['GPTPortionDescription'] == '')
count_missing_gpt = missing_gpt.sum()

print(f"CHECK 1: Rows with missing GPT Prediction")
print(f"Count: {count_missing_gpt}")
print(f"Percentage: {count_missing_gpt / len(df) * 100:.1f}%")
print("-" * 50)

# ==============================================================================
# 2. Check Ground Truth (Missing Manual Labels)
# ==============================================================================
missing_gt = df['Portion'].isna() | (df['Portion'] == '')
count_missing_gt = missing_gt.sum()

print(f"CHECK 2: Rows with missing Ground Truth (Portion)")
print(f"Count: {count_missing_gt}")
print(f"Percentage: {count_missing_gt / len(df) * 100:.1f}%")
print("-" * 50)

# ==============================================================================
# 3. Check Spelling (PortionType Unique Values)
# ==============================================================================
print(f"CHECK 3: Unique values in 'PortionType' column")
if 'PortionType' in df.columns:
    unique_types = df['PortionType'].unique()
    print(unique_types)
    
    # Also print the counts to see if 'Smallest' is truly rare
    print("\nCounts per type:")
    print(df['PortionType'].value_counts())
else:
    print("Column 'PortionType' not found in dataframe!")
print("-" * 50)

# ==============================================================================
# 4. BONUS: Simulate the Filter Logic
# ==============================================================================
# This calculates exactly how many rows survive the "valid prediction" logic
valid_mask = (
    (df['Portion'].notna()) & (df['Portion'] != '') &          # GT exists
    (df['GPTPortionDescription'].notna()) &                    # Pred exists
    (df['GPTPortionDescription'] != '')                        # Pred not empty
)
valid_count = valid_mask.sum()

print(f"FINAL CHECK: Rows with BOTH Valid GT and Valid Prediction")
print(f"Count: {valid_count}")
print(f"Does this match your table total of 884? {'YES' if valid_count == 884 else 'NO'}")
print("=" * 50)

Loading data from: ../ASA24_GPTFoodCodes_portion.csv
Total rows in file: 1715
--------------------------------------------------
CHECK 1: Rows with missing GPT Prediction
Count: 728
Percentage: 42.4%
--------------------------------------------------
CHECK 2: Rows with missing Ground Truth (Portion)
Count: 193
Percentage: 11.3%
--------------------------------------------------
CHECK 3: Unique values in 'PortionType' column
['largest' 'median' 'smallest']

Counts per type:
PortionType
largest     611
median      556
smallest    548
Name: count, dtype: int64
--------------------------------------------------
FINAL CHECK: Rows with BOTH Valid GT and Valid Prediction
Count: 884
Does this match your table total of 884? YES


In [3]:
# =============================================================================
# Calculate MAE (Mean Absolute Error) for DietAI24 on ASA24 dataset
# Per dish analysis for Energy, Fat, Carbohydrate, Protein, and Food Weight
# =============================================================================

import numpy as np

# Load the nutrition results file
df_nutrition = pd.read_csv('../data/ASA24_GPTFoodCodes_nutrition.csv')

print(f"Total rows in nutrition file: {len(df_nutrition)}")

# Exclude beverage reference images (fluid ounce portions = container reference images,
# e.g. foam cups, mugs, milk bottles used as portion size references, not food photos)
if 'Portion' in df_nutrition.columns:
    beverage_mask = df_nutrition['Portion'].str.contains('FO', na=False)
    print(f"Excluded {beverage_mask.sum()} beverage reference image rows (FO portions)")
    df_nutrition = df_nutrition[~beverage_mask].copy()
    print(f"Rows after exclusion: {len(df_nutrition)}")

# Filter to common images only (present in both old and new pipelines)
old_filenames = set(pd.read_csv('../data/archive/ASA24_GPTFoodCodes_nutrition_old.csv')['FileName'])
n_before = len(df_nutrition)
df_nutrition = df_nutrition[df_nutrition['FileName'].isin(old_filenames)].copy()
print(f"Filtered to {len(df_nutrition)} rows from common images (removed {n_before - len(df_nutrition)} new-image rows)")
print(f"Columns: {df_nutrition.columns.tolist()[:10]}...")  # Show first 10 columns

# Define the nutrients we're interested in
nutrients = ['Energy (kcal)', 'Protein (g)', 'Carbohydrate (g)', 'Total Fat (g)']

# Filter to only valid rows (where both GT and GPT predictions exist)
valid_rows = df_nutrition.copy()

# For each nutrient, filter to rows where both GT and prediction are not NaN
mae_results = {}

for nutrient in nutrients:
    gt_col = nutrient
    pred_col = f'{nutrient}_GPT'
    
    # Check if columns exist
    if gt_col not in df_nutrition.columns or pred_col not in df_nutrition.columns:
        print(f"Warning: {gt_col} or {pred_col} not found in dataframe")
        continue
    
    # Filter to valid rows for this nutrient (exclude zero predictions = GPT estimation failures)
    valid_mask = (
        df_nutrition[gt_col].notna() & 
        df_nutrition[pred_col].notna() &
        (df_nutrition[pred_col] != 0)
    )
    
    valid_data = df_nutrition[valid_mask]
    
    if len(valid_data) == 0:
        print(f"No valid data for {nutrient}")
        continue
    
    # Calculate MAE
    mae = np.abs(valid_data[gt_col] - valid_data[pred_col]).mean()
    
    mae_results[nutrient] = {
        'MAE': mae,
        'N': len(valid_data)
    }
    
# Calculate MAE for food weight
if 'CalculatedWeight' in df_nutrition.columns and 'CalculatedWeightGPT' in df_nutrition.columns:
    n_zero_weight = (df_nutrition['CalculatedWeightGPT'] == 0).sum()
    print(f"Excluded {n_zero_weight} rows with CalculatedWeightGPT=0 (GPT estimation failures)")
    weight_valid_mask = (
        df_nutrition['CalculatedWeight'].notna() & 
        df_nutrition['CalculatedWeightGPT'].notna() &
        (df_nutrition['CalculatedWeightGPT'] != 0)
    )
    weight_valid = df_nutrition[weight_valid_mask]
    
    if len(weight_valid) > 0:
        weight_mae = np.abs(weight_valid['CalculatedWeight'] - weight_valid['CalculatedWeightGPT']).mean()
        mae_results['Food weight (g)'] = {
            'MAE': weight_mae,
            'N': len(weight_valid)
        }

print("\n" + "="*70)
print("MAE Results for DietAI24 on ASA24 Dataset (Per Dish)")
print("="*70)
print(f"{'Target':<25} {'MAE (Per dish)':<20} {'N samples':<15}")
print("-"*70)

for target, results in mae_results.items():
    print(f"{target:<25} {results['MAE']:<20.2f} {results['N']:<15}")

print("="*70)

Total rows in nutrition file: 690
Excluded 0 beverage reference image rows (FO portions)
Rows after exclusion: 690
Filtered to 690 rows from common images (removed 0 new-image rows)
Columns: ['Unnamed: 0', 'FileName', 'PortionType', 'FoodCode', 'Main Food description', 'Portion', 'Multiplier', 'Rank', 'PortionCode', 'PortionSubCode']...
Excluded 10 rows with CalculatedWeightGPT=0 (GPT estimation failures)

MAE Results for DietAI24 on ASA24 Dataset (Per Dish)
Target                    MAE (Per dish)       N samples      
----------------------------------------------------------------------
Energy (kcal)             79.57                639            
Protein (g)               3.80                 617            
Carbohydrate (g)          8.72                 613            
Total Fat (g)             3.80                 620            
Food weight (g)           51.15                640            


In [8]:
df_nutrition.shape

(690, 35)

In [68]:
# Each image appears up to 3 times (largest/median/smallest) — that's expected
print("Total rows:", len(df_nutrition))
print("Unique FileNames:", df_nutrition['FileName'].nunique())
print()

# True duplicates: same FileName + PortionType
dupes = df_nutrition.duplicated(subset=['FileName', 'PortionType', 'FoodCode'], keep=False)
print(f"Duplicate (FileName, PortionType, FoodCode) rows: {dupes.sum()}")
if dupes.sum() > 0:
    display(df_nutrition[dupes][['FileName', 'PortionType', 'FoodCode', 'Energy (kcal)', 'Energy (kcal)_GPT']].sort_values(['FileName', 'PortionType', 'FoodCode']))

Total rows: 690
Unique FileNames: 434

Duplicate (FileName, PortionType, FoodCode) rows: 0


In [9]:
df_nutrition.to_csv('../data/ASA24_GPTFoodCodes_nutrition.csv', index=False)

In [2]:
df = pd.read_csv('../nutrition5k_final.csv')
df.head()

,dish_id,total_calories,total_mass,total_fat,total_carbs,total_protein,ingredients,sorted_ingredients,GPTFoodDescription,GPTFoodCode,GPTAmount,GPTAmountWeight,Energy (kcal),Protein (g),Carbohydrate (g),Total Fat (g),FoodCodeError,WeightError,IngredientNum,GPTWeight
0,dish_1567023335,138.336395,117,12.019873,5.832454,3.287284,"greek salad, garlic, salt, pecans, raspberries...","almonds, arugula, blueberries, carrot, chard, ...",curly kale leaves\narugula leaves\nSwiss chard...,curly kale leaves: 72119190; arugula leaves: 7...,curly kale leaves: 1 cup\narugula leaves: 0.5 ...,"{'curly kale leaves': 25.0, 'arugula leaves': ...",97.624,5.33878,5.29628,6.720720,0,0,6,114.80
1,dish_1561404956,97.600792,82,2.638521,4.593264,13.654346,"green onions, white rice, vinegar, ginger, par...","broccoli, caesar dressing, chicken, croutons, ...",roasted chicken thigh (skin-on)\nsautéed kale\...,roasted chicken thigh (skin-on): 24152230; sau...,roasted chicken thigh (skin-on): unknown\nsaut...,"{'roasted chicken thigh (skin-on)': nan, 'saut...",114.150,5.59750,6.84350,7.191500,0,1,3,140.00
2,dish_1563220109,184.735962,174,7.398599,22.408951,8.018723,"pepperoni pizza, salt, olive oil, squash, broc...","broccoli, olive oil, pepperoni pizza, salt, sq...",pepperoni (sliced)\nmozzarella cheese (melted)...,pepperoni (sliced): 25221250; mozzarella chees...,pepperoni (sliced): 1 slice\nmozzarella cheese...,"{'pepperoni (sliced)': 2.0, 'mozzarella cheese...",252.365,16.68025,15.51350,14.329600,0,1,6,213.00
3,dish_1561662501,158.395508,165,7.063837,21.929186,6.882387,"olive oil, pepper, corn on the cob, vinegar, c...","cherry tomatoes, corn on the cob, cucumbers, l...",cubed tofu\nbaby spinach leaves\nhalved cherry...,cubed tofu: 41420010; baby spinach leaves: 721...,cubed tofu: unknown\nbaby spinach leaves: unkn...,"{'cubed tofu': nan, 'baby spinach leaves': nan...",47.575,1.88075,9.94300,0.741875,0,2,4,151.25
4,dish_1565036821,325.539917,134,14.517972,25.940332,21.035995,"soy sauce, salt, vinegar, olive oil, beef, whi...","beef, broccoli, cheese pizza, chili, garlic, g...",melted mozzarella cheese\ntomato pizza sauce\n...,melted mozzarella cheese: 14107010; tomato piz...,"melted mozzarella cheese: 0.5 cup, melted\ntom...","{'melted mozzarella cheese': 122.0, 'tomato pi...",885.840,84.28930,23.96460,49.224100,0,2,5,405.00


In [6]:
# =============================================================================
# Calculate MAE (Mean Absolute Error) for DietAI24 on Nutrition5k dataset
# Per dish analysis for Energy, Fat, Carbohydrate, Protein, and Food Weight
# =============================================================================

print(f"Total rows in Nutrition5k results: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Define the mapping between Nutrition5k ground truth and FNDDS prediction columns
nutrient_mapping = {
    'total_calories': 'Energy (kcal)',     # GT column: total_calories -> Pred: Energy (kcal)
    'total_protein': 'Protein (g)',        # GT column: total_protein -> Pred: Protein (g)
    'total_carbs': 'Carbohydrate (g)',     # GT column: total_carbs -> Pred: Carbohydrate (g)
    'total_fat': 'Total Fat (g)',          # GT column: total_fat -> Pred: Total Fat (g)
    'total_mass': 'GPTWeight'              # GT column: total_mass -> Pred: GPTWeight
}

# Calculate MAE for each nutrient
mae_results = {}

for gt_col, pred_col in nutrient_mapping.items():
    # Check if columns exist
    if gt_col not in df.columns:
        print(f"Warning: Ground truth column {gt_col} not found")
        continue
    
    if pred_col not in df.columns:
        print(f"Warning: Prediction column {pred_col} not found")
        continue
    
    # Filter to valid rows (both GT and prediction exist)
    valid_mask = (
        df[gt_col].notna() & 
        df[pred_col].notna()
    )
    
    valid_data = df[valid_mask]
    
    if len(valid_data) == 0:
        print(f"No valid data for {gt_col}")
        continue
    
    # Calculate MAE
    mae = np.abs(valid_data[gt_col] - valid_data[pred_col]).mean()
    
    # Use readable names for display
    display_name = {
        'total_calories': 'Energy (kcal)',
        'total_protein': 'Protein (g)',
        'total_carbs': 'Carbohydrate (g)',
        'total_fat': 'Total Fat (g)',
        'total_mass': 'Food weight (g)'
    }
    
    mae_results[display_name[gt_col]] = {
        'MAE': mae,
        'N': len(valid_data)
    }

# Print formatted table
print("\n" + "="*70)
print("MAE Results for DietAI24 on Nutrition5k Dataset (Per Dish)")
print("="*70)
print(f"{'Target':<25} {'MAE (Per dish)':<20} {'N samples':<15}")
print("-"*70)

for target, results in mae_results.items():
    print(f"{target:<25} {results['MAE']:<20.2f} {results['N']:<15}")

print("="*70)

Total rows in Nutrition5k results: 1000
Columns: ['dish_id', 'total_calories', 'total_mass', 'total_fat', 'total_carbs', 'total_protein', 'ingredients', 'sorted_ingredients', 'GPTFoodDescription', 'GPTFoodCode', 'GPTAmount', 'GPTAmountWeight', 'Energy (kcal)', 'Protein (g)', 'Carbohydrate (g)', 'Total Fat (g)', 'FoodCodeError', 'WeightError', 'IngredientNum', 'GPTWeight']

MAE Results for DietAI24 on Nutrition5k Dataset (Per Dish)
Target                    MAE (Per dish)       N samples      
----------------------------------------------------------------------
Energy (kcal)             186.87               947            
Protein (g)               12.65                947            
Carbohydrate (g)          15.35                947            
Total Fat (g)             11.73                947            
Food weight (g)           121.45               1000           


In [2]:
df_source = pd.read_csv('../ASA24_GPTFoodCodes_all_portions.csv')
df_target = pd.read_csv('../ASA24_GPTFoodCodes_nutrition_metrics.csv')

merge_cols = ['retrieval_score_avg', 'retrieval_score_top1', 'retrieval_score_spread']

df_target = df_target.merge(
    df_source[['FileName', 'PortionType'] + merge_cols],
    on=['FileName', 'PortionType'],
    how='left'
)
df_target.to_csv('../ASA24_GPTFoodCodes_nutrition_metrics.csv', index=False)
df_target.head()

,FileName,PortionType,FoodCodeCommon,FC_Description,Portion,Multiplier,Rank,Link,GPTFoodDescription,GPTFoodCode,...,image_aspect_ratio,image_sharpness,color_saturation,color_diversity,edge_density,overexposure_ratio,underexposure_ratio,retrieval_score_avg,retrieval_score_top1,retrieval_score_spread
0,042108437.png,largest,91101010,"Sugar, white, granulated or lump",6 teaspoons,6.0,5,https://asa24.nci.nih.gov/images/042108437.png,"The image shows spoons/utensils, not a food item.",NaN,...,1.0,34.749918,1.173516,1.981530,0.035781,0.703047,0.000000,1.152892,0.872036,0.099508
1,042108437.png,largest,91101010,"Sugar, white, granulated or lump",6 teaspoons,6.0,5,https://asa24.nci.nih.gov/images/042108437.png,"The image shows spoons/utensils, not a food item.",NaN,...,1.0,34.749918,1.173516,1.981530,0.035781,0.703047,0.000000,1.146723,0.943052,0.077338
2,101309118.png,largest,75113000,"Lettuce, raw",4 cups,4.0,8,https://asa24.nci.nih.gov/images/101309118.png,Cooked chopped green cabbage — likely sautéed ...,['75211031'],...,1.0,93.838346,45.372695,3.985678,0.078945,0.122969,0.215938,0.975585,0.821415,0.090022
3,042908424.png,largest,74401010,Ketchup,1 packet,1.0,9,https://asa24.nci.nih.gov/images/042908424.png,Tomato ketchup — a bottled tomato-based condim...,['74401010'],...,1.0,15.942356,2.682734,0.357919,0.014570,0.963516,0.003477,1.080862,0.878266,0.091241
4,042908424.png,largest,74401010,Ketchup,1 packet,1.0,9,https://asa24.nci.nih.gov/images/042908424.png,Tomato ketchup — a bottled tomato-based condim...,['74401010'],...,1.0,15.942356,2.682734,0.357919,0.014570,0.963516,0.003477,1.070478,0.812545,0.081366


In [3]:
keep_cols = [                                                                                                                                                                                                                  
      # Identifiers                                                                                                                                                                                                              
      'FileName', 'PortionType',                                                                                                                                                                                                 
      # Ground truth & predicted calories                                                                                                                                                                                        
      'Energy (kcal)', 'Energy (kcal)_GPT',
      # Prediction-only features
      'n_items', 'desc_length_chars', 'desc_length_words', 'uncertainty_words_count',
      'retrieval_score_avg', 'retrieval_score_top1', 'retrieval_score_spread',
      # Image features
      'image_blur_score', 'image_brightness', 'image_contrast',
      'image_sharpness', 'color_saturation', 'color_diversity',
      'edge_density', 'overexposure_ratio', 'underexposure_ratio',
  ]

df_final = df_target[keep_cols].drop_duplicates(subset='FileName', keep='first').reset_index(drop=True)

In [6]:
# Only kept those rows with predicted calories (Energy (kcal)_GPT) not NA
df_final = df_final[df_final['Energy (kcal)_GPT'].notna()].copy()
df_final.reset_index(drop=True, inplace=True)

In [8]:
df_final.to_csv('../ASA24_GPTFoodCodes_metrics_final.csv', index=False)

In [40]:
df1 = pd.read_csv('../data/archive/ASA24_GPTFoodCodes_metrics_final_old.csv')
df2 = pd.read_csv('../data/archive/ASA24_GPTFoodCodes_metric_final_old1.csv')

In [41]:
df1.shape

(670, 44)

In [42]:
df2.shape

(1502, 65)

In [43]:
# Remove duplicates based on FileName and PortionType for df2
df2 = df2.drop_duplicates(subset=['FileName', 'PortionType'], keep='first').copy()
df2.reset_index(drop=True, inplace=True)
df2.shape

(876, 65)

In [44]:
cols_from_df2 = ['largest_component_ratio', 'food_mask_area_x_depth_mean', 'food_mask_area_x_depth_std', 'vit_embedding']

# Drop stale fc columns from df1 before merging fresh values from df2
df1_clean = df1.drop(columns=[c for c in cols_from_df2 if c in df1.columns], errors='ignore')

# Merge on FileName and PortionType, keeping all rows from df1 and adding columns from df2
df_merged = df1_clean.merge(
    df2[['FileName', 'PortionType'] +
        [c for c in cols_from_df2 if c in df2.columns]],
    on=['FileName', 'PortionType'],
    how='left'
)

In [7]:
df_merged.head()

,FileName,PortionType,Energy (kcal),Energy (kcal)_GPT,n_items,desc_length_chars,desc_length_words,uncertainty_words_count,retrieval_score_avg,retrieval_score_top1,...,depth_gradient_mean,depth_center_surround_diff,calorie_range,calorie_iqr,calorie_std,fc_semantic_entropy,fc_n_unique_codes,fc_entropy,fc_hierarchical_spread,fc_sampled_codes
0,042108437.png,largest,101.052,101.052,0,49,8,0,1.152892,0.872036,...,63.249645,126.866130,NaN,NaN,NaN,2.3219,0.0,0.000,3.0,"[null, null, null, null, null]"
1,042108437.png,largest,101.052,101.052,0,49,8,0,1.152892,0.872036,...,63.249645,126.866130,NaN,NaN,NaN,2.3219,0.0,0.000,3.0,"[null, null, null, null, null]"
2,101309118.png,largest,19.600,4.900,1,77,11,2,0.975585,0.821415,...,158.212654,431.616292,2.450,0.000,0.98000,-0.0000,1.0,-0.000,0.0,"[""75211031"", ""75211031"", ""75211031"", ""75211031..."
3,092607118.png,largest,374.400,187.200,1,31,5,0,1.127739,0.927607,...,106.865852,172.178924,146.016,73.008,60.39598,0.9710,3.0,1.371,1.3,"[""51184000"", ""54408017"", ""51184000"", ""54301010..."
4,113096.png,largest,563.040,563.040,1,67,10,1,1.217577,1.024695,...,32.015651,-8.281563,0.000,0.000,0.00000,2.3219,1.0,-0.000,1.2,"[""95220000"", ""95220000"", ""95220000"", ""95220000..."


In [45]:
df_merged.shape

(670, 48)

In [46]:
# calculate log predicted kcal using the column 'Energy (kcal)_GPT'
df_merged['log_Energy_kcal_GPT'] = np.log(df_merged['Energy (kcal)_GPT'] + 1)  # add 1 to avoid log(0)

In [47]:
df_merged.to_csv('../data/archive/ASA24_GPTFoodCodes_metrics_final_old2.csv', index=False)

In [32]:
df1 = pd.read_csv('../data/archive/ASA24_GPTFoodCodes_metrics_final_old.csv')
df2 = pd.read_csv('../data/archive/ASA24_GPTFoodCodes_uncertainty_old_copy.csv')

In [33]:
df1.shape

(670, 44)

In [34]:
fc_cols = ['fc_n_unique_codes', 'fc_entropy', 'fc_hierarchical_spread', 'fc_sampled_codes']
df2_dedup = df2.drop_duplicates(subset=['FileName', 'PortionType'], keep='first')

# Drop stale fc columns from df1 and replace with fresh values from df2
df1_clean = df1.drop(columns=[c for c in fc_cols if c in df1.columns], errors='ignore')
df1_updated = df1_clean.merge(
    df2_dedup[['FileName', 'PortionType'] + [c for c in fc_cols if c in df2_dedup.columns]],
    on=['FileName', 'PortionType'],
    how='left'
)

In [36]:
df1_updated.head(10)

,FileName,PortionType,Energy (kcal),Energy (kcal)_GPT,n_items,desc_length_chars,desc_length_words,uncertainty_words_count,retrieval_score_avg,retrieval_score_top1,...,depth_gradient_mean,depth_center_surround_diff,calorie_range,calorie_iqr,calorie_std,fc_semantic_entropy,fc_n_unique_codes,fc_entropy,fc_hierarchical_spread,fc_sampled_codes
0,042108437.png,largest,101.052,101.052,0,49,8,0,1.152892,0.872036,...,63.249645,126.866130,NaN,NaN,NaN,2.3219,0.0,0.000,3.0,"[null, null, null, null, null]"
1,042108437.png,largest,101.052,101.052,0,49,8,0,1.152892,0.872036,...,63.249645,126.866130,NaN,NaN,NaN,2.3219,0.0,0.000,3.0,"[null, null, null, null, null]"
2,101309118.png,largest,19.600,4.900,1,77,11,2,0.975585,0.821415,...,158.212654,431.616292,2.450,0.000,0.98000,-0.0000,1.0,-0.000,0.0,"[""75211031"", ""75211031"", ""75211031"", ""75211031..."
3,092607118.png,largest,374.400,187.200,1,31,5,0,1.127739,0.927607,...,106.865852,172.178924,146.016,73.008,60.39598,0.9710,3.0,1.371,1.3,"[""51184000"", ""54408017"", ""51184000"", ""54301010..."
4,113096.png,largest,563.040,563.040,1,67,10,1,1.217577,1.024695,...,32.015651,-8.281563,0.000,0.000,0.00000,2.3219,1.0,-0.000,1.2,"[""95220000"", ""95220000"", ""95220000"", ""95220000..."
5,113096.png,largest,563.040,563.040,1,67,10,1,1.217577,1.024695,...,32.015651,-8.281563,0.000,0.000,0.00000,2.3219,1.0,-0.000,1.2,"[""95220000"", ""95220000"", ""95220000"", ""95220000..."
6,113096.png,largest,563.040,563.040,1,67,10,1,1.217577,1.024695,...,32.015651,-8.281563,0.000,0.000,0.00000,2.3219,1.0,-0.000,1.2,"[""95220000"", ""95220000"", ""95220000"", ""95220000..."
7,113096.png,largest,563.040,563.040,1,67,10,1,1.217577,1.024695,...,32.015651,-8.281563,0.000,0.000,0.00000,2.3219,1.0,-0.000,1.2,"[""95220000"", ""95220000"", ""95220000"", ""95220000..."
8,113096.png,largest,563.040,563.040,1,67,10,1,1.217577,1.024695,...,32.015651,-8.281563,0.000,0.000,0.00000,2.3219,1.0,-0.000,1.2,"[""95220000"", ""95220000"", ""95220000"", ""95220000..."
9,113096.png,largest,563.040,563.040,1,67,10,1,1.217577,1.024695,...,32.015651,-8.281563,0.000,0.000,0.00000,2.3219,1.0,-0.000,1.2,"[""95220000"", ""95220000"", ""95220000"", ""95220000..."


In [35]:
df1_updated.shape

(670, 44)

In [38]:
df1_updated.to_csv('../data/archive/ASA24_GPTFoodCodes_metric_final_old1.csv', index=False)

In [70]:
df1 = pd.read_csv('../data/ASA24_GPTFoodCodes_metrics_final.csv')
df2 = pd.read_csv('../data/ASA24_GPTFoodCodes_nutrition.csv')

In [71]:
# check unique FileName + PortionType + FoodCode combinations in both dataframes, and filter df1 to only those combinations that exist in df2
df2['FileName'] = df2['FileName'].str.lower().str.strip()
df2['PortionType'] = df2['PortionType'].str.lower().str.strip()
df2['FoodCode'] = df2['FoodCode'].astype(str).str.strip()
df2['UniqueKey'] = df2['FileName'] + '|' + df2['PortionType'] + '|' + df2['FoodCode']
df1['UniqueKey'] = df1['FileName'].str.lower().str.strip() + '|' + df1['PortionType'].str.lower().str.strip() + '|' + df1['FoodCode'].astype(str).str.strip()
df1_filtered = df1[df1['UniqueKey'].isin(df2['UniqueKey'])].copy()
df1_filtered.reset_index(drop=True, inplace=True)

In [73]:
df1_filtered.shape

(690, 111)

In [74]:
df1_filtered.to_csv('../data/ASA24_GPTFoodCodes_metrics_final.csv', index=False)